# BRENDAN REED   |   brreed@andrew.cmu.edu

# Question 1:
I would expect the average win rate at TicTacToe for Player 1 and Player 2 selecting completely at random to be **approximately 40%**. In my research I found that a normal 3x3 game of TicTacToe results in a draw roughly 18% of the time. If two players are moving completely randomly, minus this 18% average draw rate, each player would have roughly the same probability to win.
Our HW2 doc states that a 4x4 grid results in far fewer draws than 3x3, however this is not what my gameplay showed throughout simulation experiments. In fact, the draw rate jumped up to about 40%. After that, players 1 and 2 each had approximately 30% probability to win a game not resulting in a draw.

https://www.r-bloggers.com/2017/03/tic-tac-toe-simulation-random-moves/

In [13]:
import numpy as np
import random
import math
import pandas as pd

# Class to represent the 4x4 Tic-Tac-Toe game
class TicTacToe:
    def __init__(self):
          # Initialize empty board
          self.board = np.zeros((4, 4), dtype=int)      # Create an np array of 4x4 zeroes to set the board
          self.current_player = 1                       # Set Player1 value as 1.
          self.game_over = False
          self.winner = None

          self.available_moves()


    # Set a variable where function knows all EMPTY positions on the board
    def available_moves(self):
        free_spaces = list(zip(*np.where(self.board == 0)))     # Used Gen AI to help explain the zip() function and how it is applicable here. Gen AI also troubleshooted this line ONLY by telling me I needed to add the '*' before 'np'
        return free_spaces


     # Current player places move on the board based on available free_spaces.
    def make_move(self, free_spaces):
        if free_spaces:
            row, col = random.choice(free_spaces)           # From free_spaces choose random row,col position.
            self.board[row][col] = self.current_player      # At the selected free space, place an integer 1 or -1 to indicate which player now occupies that spot.
            self.current_player *= -1                       # Iterate to the next player. Player 1 becomes Player -1 and vice versa.


    # Check if a winner exists in the current board state (on this turn).
    def check_winner(self):
        win = 4                     # A win is classified as 4 [in a row].
        for i in range(4):
            if abs(sum(self.board[i, :])) == win:                       # Gen AI helped polish this function and debug through using abs()
                self.winner = int(np.sign(sum(self.board[i, :])))       # Gen AI helped write this function and use np.sign() correctly.
                self.game_over = True
                return self.winner

            if abs(sum(self.board[:, i])) == win:
                self.winner = int(np.sign(sum(self.board[:, i])))
                self.game_over = True
                return self.winner

        NW_SE_diagonal = sum(self.board[i, i] for i in range(4))            # Check diagonals of the board for four in a row.
        NE_SW_diagonal = sum(self.board[i, 3 - i] for i in range(4))

        for diag in [NW_SE_diagonal, NE_SW_diagonal]:
            if abs(diag) == win:
                self.winner = int(np.sign(diag))           # Winner is set as either 1 or -1, whichever occupies the space of four in a row on either diagonal.
                self.game_over = True
                return self.winner

        # Check for draw (no empty cells remaining)
        if not np.any(self.board == 0):
            self.game_over = True
            return 0

        # Winner not declared yet? (on this turn) = game is still ongoing. Loop continues until a condition above is filled.
        return None


    # Play a full game with both players moving randomly.
    def play_game(self):
        while not self.game_over:
            empty_cells = list(zip(*np.where(self.board == 0)))
            if empty_cells:
                row, col = random.choice(empty_cells)
                self.board[row][col] = self.current_player
                self.current_player *= -1
            result = self.check_winner()
        return result


# Play the game.
def run_simulations(n=1000):
    results = {"X_wins": 0, "O_wins": 0, "Draws": 0}        # Reset game score each run.


    for i in range(n):
        game = TicTacToe()
        result = game.play_game()

        # Record results.
        if result == 1:
            results["X_wins"] += 1
            outcome = "X_wins"
        elif result == -1:
            results["O_wins"] += 1
            outcome = "O_wins"
        else:
            results["Draws"] += 1
            outcome = "Draw"

    # Print summary
    print(f"Results over {n} simulations:")
    print(f"  X Wins: {results['X_wins']} ({results['X_wins']/n*100}%)")
    print(f"  O Wins: {results['O_wins']} ({results['O_wins']/n*100}%)")
    print(f"  Draws:  {results['Draws']} ({results['Draws']/n*100}%)")

    return results

# Run simulations of random tic-tac-toe game 1000 times.
results = run_simulations(1000)

Results over 1000 simulations:
  X Wins: 293 (29.299999999999997%)
  O Wins: 265 (26.5%)
  Draws:  442 (44.2%)


In [ ]:
# Monte Carlo Tree Search Node
class MCTS_Game:
    def __init__(self, state, parent=None):
        self.state = state                              # Current board state
        self.parent = parent                            # Parent node (Max)
        self.children = []                              # Child nodes (Min)
        self.wins = 0                                   # Wins for node n
        self.visits = 0                                 # Number of times this node was visited
        self.untried_moves = state.available_moves()    # Moves not yet expanded

    # Checks if a node has been fully explored (ends in win, loss, or draw)
    def is_fully_expanded(self):
        return len(self.untried_moves) == 0

    # Determine optimal moves that maximize below formula. Use exploration parameter sqrt(2)
    def best_child(self, exploration_weight=1.4):
        # From lecture slides 8
        # formula: (wins/nodes) + (C * sqrt(ln(parent) / child))
        return max(
            self.children,                      # Gen AI helped troubleshoot parts this function. See attached.
            key=lambda child: (child.wins / child.visits) + exploration_weight * math.sqrt(math.log(self.visits) / child.visits)
)


# Monte Carlo Tree Search Algorithm
class MCTS:
    def __init__(self, itermax=1000, exploration_weight=1.4):
        self.itermax = itermax                                  # Simulation budget per move
        self.exploration_weight = exploration_weight            # Exploration weight, C sqrt(2)


    # FUNCTION FOR STEP ONE: SELECTION - Start from the root (current state) and select successive child nodes until a leaf node is reached.
    def select(self, node):
        while not node.state.game_over:
            if not node.is_fully_expanded():
                return node                         # This node still has untried moves. return to be used in next steps.
            else:
                node = node.best_child(self.exploration_weight)
        return node


    # FUNCTION FOR STEP TWO: EXPANSION - Unless the lead node ends the game, create one or more child nodes and select one.
    def expand(self, node):
        move = node.untried_moves.pop()             # Take an untried move

        # Copy game state to each node.
        new_state = TicTacToe()
        new_state.board = node.state.board.copy()                   # Interfaced with Gen AI to write these two lines about copying the current_player and state of the board.
        new_state.current_player = node.state.current_player
        new_state.game_over = node.state.game_over
        new_state.winner = node.state.winner

        # Apply move to node state
        row, col = move
        new_state.board[row][col] = new_state.current_player
        new_state.current_player *= -1
        new_state.check_winner()

        child_node = MCTS_Game(state=new_state, parent=node)
        node.children.append(child_node)
        return child_node


    # FUNCTION FOR STEP THREE: SIMULATION - Play a random game from the node chosen in expansion phase until a result is achieved.
    def simulate(self, state):

        # Copy so simulation doesn't affect the actual tree during this playthrough
        sim_board = state.board.copy()
        sim_player = state.current_player
        sim_game_over = state.game_over
        sim_winner = state.winner

        while not sim_game_over:
            free_spaces = list(zip(*np.where(sim_board == 0)))
            if not free_spaces:
                break                   # Draw or winner.
            row, col = random.choice(free_spaces)
            sim_board[row][col] = sim_player
            sim_player *= -1

            # Check winner after each move DURING SIMULATION. Similar to random game above.
            wins = 4
            sim_winner = None
            for i in range(4):
                for line in [sim_board[i, :], sim_board[:, i]]:
                    s = sum(line)
                    if abs(s) == wins:
                        sim_winner = int(np.sign(s))
                        sim_game_over = True
            # Check diagonals
            main_diag = sum(sim_board[i, i] for i in range(4))
            anti_diag = sum(sim_board[i, 3 - i] for i in range(4))
            for diag in [main_diag, anti_diag]:
                if abs(diag) == wins:
                    sim_winner = int(np.sign(diag))
                    sim_game_over = True
            if not np.any(sim_board == 0):
                sim_game_over = True

        # Score MCTS player
        if sim_winner == 1:
            return 1.0          # Win
        elif sim_winner == -1:
            return 0.0          # Loss
        else:
            return 0.5          # Draw


    # FUNCTION FOR STEP FOUR: BACKPROPOGATION - Update the current move sequence with the simulation result.
    def backpropagate(self, node, result):
        while node is not None:
            node.visits += 1
            node.wins += result
            node = node.parent


    # CREATE ROOT node from current game state
    # RUNS ACTUAL MCTS FUNCTIONS DEFINED ABOVE
    def search(self, state):
        root = MCTS_Game(state=state)

        # SELECTION — traverse the tree to find node.
        # CREATE NODE
        for _ in range(self.itermax):
            node = self.select(root)

            # EXPANSION — if game state is not ended at a node n, add new child.
            # CREATE EXPANDED NODE
            if not node.state.game_over:
                node = self.expand(node)

            # SIMULATION — play a game from new node
            # CREATE NEW GAME
            result = self.simulate(node.state)

            # BACKPROPAGATION — update win/visit counts
            self.backpropagate(node, result)

        # Return the move corresponding to the most visited child of root
        best = max(root.children, key=lambda child: child.visits)
        return best.state

# Usually Colab lets you use functions from other cells but this time it wasn't working.
def play_game():
    game = TicTacToe()
    mcts = MCTS(itermax=1000)  # Adjust iterations as needed

    while not game.game_over:
        if game.current_player == 1:
            # MCTS chooses the next state
            game = mcts.search(game)
        else:
            # Random player picks a random available move
            empty_cells = game.available_moves()
            if empty_cells:
                row, col = random.choice(empty_cells)
                game.board[row][col] = game.current_player
                game.current_player *= -1
                game.check_winner()

    return game.winner


# Main execution - run multiple games
if __name__ == "__main__":
    num_games = 500
    results = {"X_wins": 0, "O_wins": 0, "Draws": 0}

    for i in range(num_games):
        winner = play_game()
        if winner == 1:
            results["X_wins"] += 1
        elif winner == -1:
            results["O_wins"] += 1
        else:
            results["Draws"] += 1

    print(f"\nResults over {num_games} games (MCTS=X vs Random=O):")
    print(f"  MCTS  Wins: {results['X_wins']} ({results['X_wins']/num_games*100}%)")
    print(f"  Random Wins: {results['O_wins']} ({results['O_wins']/num_games*100}%)")
    print(f"  Draws:       {results['Draws']} ({results['Draws']/num_games*100}%)")

# QUESTION 3:

MCTS performs wildly better at 4x4 TicTacToe than any random play-style achieved in the previous implementation. Draw rate drops to nearly zero, and in many of my simulations, the random player frequently did not win a single game after 500 play throughs. There are exceptions to this, however it is impressive to see this algorithm succeed so wildly across a very high number of trials.

# QUESTION 4:



In [ ]:
import matplotlib.pyplot as plt

c_values            = [0.0, 0.2, 0.5, 0.7, 1.0, 1.4, 1.8, 2.5, 3.5]
num_games           = 125           # games per C value
mcts_iter_max       = 200          # iterations per move
# ─────────────────────────────────────────────────────────────────────────────

def play_game_2(c):
    mcts   = MCTS(itermax=mcts_iter_max, exploration_weight=c)      # Calls functions from above
    game   = TicTacToe()

    while not game.game_over:
        if game.current_player == 1:           # MCTS move
            game = mcts.search(game)
        else:                                  # Random move
            empty = list(zip(*np.where(game.board == 0)))
            if empty:
                r, col = random.choice(empty)
                game.board[r][col] = game.current_player
                game.current_player *= -1
                game.check_winner()

    return game.winner

win_rates = []

for c in c_values:
    wins = sum(1 for _ in range(num_games) if play_game_2(c) == 1)
    rate = wins / num_games * 100
    win_rates.append(rate)
    print(f"C = {c}  MCTS win rate: {rate:.1f}%")

# MatPlot plot for c
best_c   = c_values[win_rates.index(max(win_rates))]
best_wr  = max(win_rates)

plt.plot(c_values, win_rates, marker="o")
plt.axvline(best_c, color="red", label=f"Best C")
plt.scatter([best_c], [best_wr], color="red")

plt.xlabel("Exploration Weight C")
plt.ylabel("Win Rate (%)")

plt.show()
print(f"\nBest exploration weight: C = {best_c}")